# PlantCLEF 2015 Upper Bound

This notebook is intentionally separated from the main training notebooks. It trains on an adaptation split carved out of the official test package and evaluates on the remaining holdout split. Metrics from this notebook are **not valid unbiased test metrics** and should only be used as a diagnostic/upper-bound experiment.

The saved checkpoints are named `final_...` so they cannot be confused with normal S-CNN(A/B) checkpoints.

## 1. Runtime Check

Use a GPU runtime. This notebook fine-tunes both VGG16 S-CNN stages and repeatedly evaluates the two-stage ranking.

In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Clone Or Update Project

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[ml]'], check=True)
src_path = str(PROJECT_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')

## 3. Mount Google Drive

In [ ]:
import os
import shutil
from pathlib import Path

if os.path.ismount('/content/drive') and Path('/content/drive/MyDrive').exists():
    print('Google Drive is already mounted at /content/drive')
else:
    if Path('/content/drive').exists() and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)


## 4. Restore LeafScan Train/Test Data

Expected archives on Drive:

- `/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz`
- `/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz`

The training metadata is narrowed to the official 60 test species so reference selection remains comparable to the paper60 protocol.

In [ ]:
%%bash
set -euo pipefail
trap 'echo "FAILED at line $LINENO: $BASH_COMMAND" >&2' ERR
export PYTHONUNBUFFERED=1
cd /content/diploma

ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz

if [ ! -f "$ARCHIVE" ]; then
  echo "Missing LeafScan training archive: $ARCHIVE" >&2
  exit 2
fi
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing LeafScan test archive: $TEST_ARCHIVE" >&2
  exit 3
fi

rm -rf data/plantclef2015
mkdir -p data/plantclef2015

echo "extracting training archive: $ARCHIVE"
tar -xzf "$ARCHIVE" -C data/plantclef2015
test -f data/plantclef2015/leafscan/metadata.csv
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv

echo "extracting test archive: $TEST_ARCHIVE"
rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv

python - <<'PY2'
import csv
from pathlib import Path

with open('data/plantclef2015/leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
with open('data/plantclef2015/test_leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    test_rows = list(csv.DictReader(file))

test_species = {row['species'] for row in test_rows}
paper60_rows = [row for row in source_rows if row['species'] in test_species]
source_species = {row['species'] for row in source_rows}
missing_in_train = sorted(test_species - source_species)
if missing_in_train:
    raise RuntimeError(f'Missing test species in train metadata: {missing_in_train}')

fieldnames = list(source_rows[0].keys())
with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(paper60_rows)

print('leafscan source rows:', len(source_rows))
print('paper60 train rows:', len(paper60_rows))
print('paper60 train species:', len({row['species'] for row in paper60_rows}))
print('paper60 train genera:', len({row['genus'] for row in paper60_rows}))
print('official test rows:', len(test_rows))
print('official test species:', len(test_species))
print('official test genera:', len({row['genus'] for row in test_rows}))
PY2

## 5. Locate Best VGG16 S-CNN Checkpoints

Set `MANUAL_CHECKPOINTS` if you want a specific run. Otherwise the newest matching checkpoints under `/content/drive/MyDrive/diploma_checkpoints` are copied locally.

In [ ]:
from pathlib import Path
import json
import inspect
import shutil as shutil_module

MANUAL_CHECKPOINTS = {
    'genus_best': '',
    'species_best': '',
}
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/diploma_checkpoints')
LOCAL_CHECKPOINT_ROOT = Path('/content/diploma/checkpoints')
LOCAL_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

PATTERNS = {
    'genus_best': [
        'leafscan_vgg16/**/scnn_genus_vgg16_best.pt',
        '**/scnn_genus_vgg16_best.pt',
    ],
    'species_best': [
        'leafscan_vgg16/**/scnn_species_vgg16_best.pt',
        '**/scnn_species_vgg16_best.pt',
    ],
}
LOCAL_NAMES = {
    'genus_best': 'scnn_genus_vgg16_best.pt',
    'species_best': 'scnn_species_vgg16_best.pt',
}


def newest_checkpoint(patterns):
    candidates = []
    for pattern in patterns:
        candidates.extend(DRIVE_CHECKPOINT_ROOT.glob(pattern))
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No checkpoint found under {DRIVE_CHECKPOINT_ROOT} for patterns={patterns}')
    return candidates[0]

selected = {}
for key, patterns in PATTERNS.items():
    source = Path(MANUAL_CHECKPOINTS[key]) if MANUAL_CHECKPOINTS[key] else newest_checkpoint(patterns)
    if not source.exists():
        raise FileNotFoundError(source)
    target = LOCAL_CHECKPOINT_ROOT / LOCAL_NAMES[key]
    shutil_module.copy2(source, target)
    selected[key] = str(target)
    print(f'{key}: {source} -> {target} ({target.stat().st_size} bytes)')

paths_file = Path('/content/diploma/.upper_bound_checkpoint_paths.json')
paths_file.write_text(json.dumps(selected, indent=2), encoding='utf-8')
print('Saved checkpoint map:', paths_file)

## 6. Build Official-Test Adapt/Holdout Split

This is the intentionally leaky part. Each species with at least two test images keeps at least one reference/adaptation image and at least one holdout image. Single-image species are placed into `adapt` because they cannot be evaluated without having no same-species reference.

In [ ]:
import csv
import random
from collections import Counter, defaultdict
from pathlib import Path

TEST_METADATA = Path('/content/diploma/data/plantclef2015/test_leafscan_metadata.csv')
ADAPT_METADATA = Path('/content/diploma/data/plantclef2015/test_leafscan_adapt_holdout_metadata.csv')
SEED = 20260526
ADAPT_RATIO = 0.60
rng = random.Random(SEED)

with TEST_METADATA.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
fieldnames = list(rows[0].keys())
if 'split' not in fieldnames:
    fieldnames.append('split')

by_species = defaultdict(list)
for row in rows:
    by_species[row['species']].append(row)

split_rows = []
for species in sorted(by_species):
    items = sorted(by_species[species], key=lambda row: row['image_path'])
    rng.shuffle(items)
    n = len(items)
    if n == 1:
        adapt_count = 1
    else:
        adapt_count = max(1, int(round(n * ADAPT_RATIO)))
        adapt_count = min(adapt_count, n - 1)
    for index, row in enumerate(items):
        row = dict(row)
        row['split'] = 'adapt' if index < adapt_count else 'holdout'
        split_rows.append(row)

ADAPT_METADATA.parent.mkdir(parents=True, exist_ok=True)
with ADAPT_METADATA.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sorted(split_rows, key=lambda row: row['image_path']))

print('saved:', ADAPT_METADATA)
print('split counts:', dict(Counter(row['split'] for row in split_rows)))
print('adapt species:', len({row['species'] for row in split_rows if row['split'] == 'adapt'}))
print('holdout species:', len({row['species'] for row in split_rows if row['split'] == 'holdout'}))
print('adapt genera:', len({row['genus'] for row in split_rows if row['split'] == 'adapt'}))
print('holdout genera:', len({row['genus'] for row in split_rows if row['split'] == 'holdout'}))
print('smallest species counts:', Counter(row['species'] for row in split_rows).most_common()[-10:])

## 7. Fine-Tune/Evaluate Helpers

The loop below starts from the saved best S-CNN(A/B) weights, fine-tunes on `adapt`, evaluates on `holdout`, and saves the latest artifacts as `final_*`. Again, these checkpoints are diagnostic and adapted.

In [ ]:
import copy
import csv
import inspect
import json
import shutil
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/diploma')
src_path = str(PROJECT_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import yaml

from plant_classifier.data import filter_records_by_split, load_metadata_csv
from plant_classifier.inference.scnn import TwoStageSiamesePredictor
from plant_classifier.models.siamese import BackboneSpec, build_siamese_network
from plant_classifier.training.genus_eval import evaluate_genus_retrieval, select_genus_references
from plant_classifier.training.loop import train_siamese_with_dynamic_pairs
from plant_classifier.training.species_eval import (
    build_reference_embeddings,
    evaluate_species_retrieval,
    select_species_references,
)
from plant_classifier.training.validation import validate_records_exist

CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints'
OUTPUT_DIR = Path('/content/drive/MyDrive/diploma_checkpoints/upper_bound')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CONFIG_PATH = PROJECT_DIR / 'configs/leafscan_paper60_training.yaml'
with TRAIN_CONFIG_PATH.open('r', encoding='utf-8') as file:
    BASE_CONFIG = yaml.safe_load(file)

ADAPT_CONFIG = copy.deepcopy(BASE_CONFIG)
ADAPT_CONFIG['dataset'] = {
    'name': 'PlantCLEF2015LeafScanOfficialTestAdaptHoldout',
    'root': 'data/plantclef2015/test_leafscan/leafscan',
    'metadata': 'data/plantclef2015/test_leafscan_adapt_holdout_metadata.csv',
    'image_column': 'image_path',
    'family_column': 'family',
    'genus_column': 'genus',
    'species_column': 'species',
}
ADAPT_CONFIG['training'] = dict(ADAPT_CONFIG['training'])
ADAPT_CONFIG['training']['split'] = 'adapt'
ADAPT_CONFIG['training']['epochs'] = 2
ADAPT_CONFIG['training']['max_iterations'] = 256
ADAPT_CONFIG['training']['learning_rate'] = 1e-4
ADAPT_CONFIG['training']['checkpoint_every_epochs'] = {'genus': 0, 'species': 0}
ADAPT_CONFIG['evaluation'] = {'enabled': False}
ADAPT_CONFIG_PATH = PROJECT_DIR / 'configs/leafscan_upper_bound.yaml'
with ADAPT_CONFIG_PATH.open('w', encoding='utf-8') as file:
    yaml.safe_dump(ADAPT_CONFIG, file, sort_keys=False, allow_unicode=True)
print('wrote:', ADAPT_CONFIG_PATH)


def load_records_from_config(config: dict):
    dataset = config['dataset']
    return load_metadata_csv(
        metadata_path=Path(dataset['metadata']),
        dataset_root=Path(dataset['root']),
        image_column=dataset['image_column'],
        family_column=dataset['family_column'],
        genus_column=dataset['genus_column'],
        species_column=dataset['species_column'],
    )

records = load_records_from_config(ADAPT_CONFIG)
adapt_records = filter_records_by_split(records, 'adapt')
holdout_records = filter_records_by_split(records, 'holdout')
validate_records_exist(adapt_records)
validate_records_exist(holdout_records)
print('adapt records:', len(adapt_records))
print('holdout records:', len(holdout_records))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_SPEC = BackboneSpec(name=BASE_CONFIG['model']['backbone'], pretrained=False)
IMAGE_SIZE = int(BASE_CONFIG['views']['global']['image_size'])
CROP_SIZE = int(BASE_CONFIG['views']['local']['crop_size'])
CROP_POSITION = str(BASE_CONFIG['views']['local'].get('crop_position', 'center'))
PREPROCESSING = bool(BASE_CONFIG.get('preprocessing', {}).get('enabled', False))


def load_siamese(checkpoint_path: Path):
    model = build_siamese_network(MODEL_SPEC).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    return model


def train_siamese_with_dynamic_pairs_compat(**kwargs):
    params = inspect.signature(train_siamese_with_dynamic_pairs).parameters
    supported = {key: value for key, value in kwargs.items() if key in params}
    dropped = sorted(set(kwargs) - set(supported))
    if dropped:
        print('train_siamese_with_dynamic_pairs does not support:', ', '.join(dropped))
    return train_siamese_with_dynamic_pairs(**supported)


def fine_tune_stage(input_checkpoint: Path, output_checkpoint: Path, stage: str, round_seed: int) -> Path:
    model = load_siamese(input_checkpoint)
    train_siamese_with_dynamic_pairs_compat(
        model=model,
        records=adapt_records,
        taxonomic_level=stage,
        view='global' if stage == 'genus' else 'local',
        checkpoint_path=output_checkpoint,
        positive_count=int(BASE_CONFIG['pair_sampling']['positive_per_epoch'][stage]),
        negative_count=int(BASE_CONFIG['pair_sampling']['negative_per_epoch'][stage]),
        hard_negative_ratio=float(BASE_CONFIG['pair_sampling'].get('hard_negative_ratio', {}).get(stage, 0.0)),
        targeted_negative_ratio=0.0,
        targeted_negative_label_pairs=[],
        pair_sampling_strategy=str(BASE_CONFIG['pair_sampling'].get('strategy', 'pair_uniform')),
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        batch_size=16,
        epochs=int(ADAPT_CONFIG['training']['epochs']),
        learning_rate=float(ADAPT_CONFIG['training']['learning_rate']),
        momentum=float(BASE_CONFIG['training']['momentum']),
        lr_decay_step=0,
        lr_decay_gamma=1.0,
        max_iterations=int(ADAPT_CONFIG['training']['max_iterations']),
        num_workers=2,
        seed=round_seed,
        eval_fn=None,
        progress_every=20,
        checkpoint_every_epochs=0,
    )
    return output_checkpoint


def build_reference_embeddings_compat(*, genus_model, species_model, references, image_size, crop_size, crop_position, preprocessing, device):
    kwargs = {
        'genus_model': genus_model,
        'species_model': species_model,
        'references': references,
        'image_size': image_size,
        'crop_size': crop_size,
        'preprocessing': preprocessing,
        'device': device,
    }
    if 'crop_position' in inspect.signature(build_reference_embeddings).parameters:
        kwargs['crop_position'] = crop_position
    return build_reference_embeddings(**kwargs)


def build_two_stage_predictor_compat(**kwargs):
    params = inspect.signature(TwoStageSiamesePredictor.__init__).parameters
    supported = {key: value for key, value in kwargs.items() if key in params}
    dropped = sorted(set(kwargs) - set(supported))
    if dropped:
        print('TwoStageSiamesePredictor does not support:', ', '.join(dropped))
    return TwoStageSiamesePredictor(**supported)


@torch.inference_mode()
def evaluate_pair(genus_checkpoint: Path, species_checkpoint: Path, *, reference_seed: int = 42):
    genus_model = load_siamese(genus_checkpoint).eval()
    species_model = load_siamese(species_checkpoint).eval()

    genus_references = select_genus_references(
        adapt_records,
        references_per_genus=6,
        seed=reference_seed,
    )
    species_references = select_species_references(
        adapt_records,
        references_per_species=6,
        allowed_species={record.species for record in holdout_records},
    )
    validate_records_exist(genus_references)
    validate_records_exist(species_references)

    genus_result = evaluate_genus_retrieval(
        model=genus_model,
        references=genus_references,
        queries=holdout_records,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        top_ks=(5, 15, 30),
        device=DEVICE,
        preprocessing=PREPROCESSING,
        score_mode='l1',
    )

    genus_embeddings = build_reference_embeddings_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=genus_references,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        device=DEVICE,
    )
    species_embeddings = build_reference_embeddings_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=species_references,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        device=DEVICE,
    )
    predictor = build_two_stage_predictor_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=species_embeddings,
        genus_references=genus_embeddings,
        genus_candidates=30,
        top_k=5,
        image_size=IMAGE_SIZE,
        local_crop_size=CROP_SIZE,
        local_crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        genus_score_mode='l1',
        species_score_mode='l1',
        species_aggregation='max',
        genus_candidate_mode='unique',
        genus_weight_mode='score',
        device=str(DEVICE),
    )
    species_result = evaluate_species_retrieval(
        predictor=predictor,
        queries=holdout_records,
        top_ks=(1, 3, 5),
        ranking_limit=len({record.species for record in species_references}),
    )
    return genus_result, species_result


def print_metrics(label: str, genus_result, species_result):
    print(f'=== {label} ===')
    print(
        'genus:',
        ' '.join(f'top{k}={genus_result.accuracies[k]:.3f}' for k in genus_result.top_ks),
        f'queries={genus_result.queries}',
    )
    print(
        'species:',
        ' '.join(f'top{k}={species_result.accuracies[k]:.3f}' for k in species_result.top_ks),
        f'S={species_result.plantclef_s:.3f}',
        f'queries={species_result.queries}',
    )

## 8. Baseline Holdout Evaluation Before Adaptation

This checks the copied best checkpoints before any upper-bound updates.

In [ ]:
GENUS_START = CHECKPOINT_DIR / 'scnn_genus_vgg16_best.pt'
SPECIES_START = CHECKPOINT_DIR / 'scnn_species_vgg16_best.pt'

genus_result, species_result = evaluate_pair(GENUS_START, SPECIES_START)
print_metrics('before upper-bound', genus_result, species_result)

## 9. Upper-Bound Loop

The stopping targets are applied to the **holdout split from the official test package**, so this is still not an unbiased metric. The loop saves the latest adapted weights after every round and copies the final pair to Drive.

In [ ]:
TARGET_HOLDOUT_TOP1 = 0.58
TARGET_HOLDOUT_GENUS_TOP30 = 0.90
MAX_ROUNDS = 6
ROUND_EPOCHS = 2
ADAPT_CONFIG['training']['epochs'] = ROUND_EPOCHS

current_genus = GENUS_START
current_species = SPECIES_START
rows = []

for round_index in range(1, MAX_ROUNDS + 1):
    print(f'\n### adaptation round {round_index}/{MAX_ROUNDS}')
    next_genus = CHECKPOINT_DIR / f'final_scnn_genus_vgg16_round{round_index}.pt'
    next_species = CHECKPOINT_DIR / f'final_scnn_species_vgg16_round{round_index}.pt'

    current_genus = fine_tune_stage(current_genus, next_genus, 'genus', SEED + 1000 * round_index)
    shutil.copy2(current_genus, OUTPUT_DIR / current_genus.name)
    best_genus = current_genus.with_name(f'{current_genus.stem}_best{current_genus.suffix}')
    if best_genus.exists():
        shutil.copy2(best_genus, OUTPUT_DIR / best_genus.name)

    current_species = fine_tune_stage(current_species, next_species, 'species', SEED + 1000 * round_index + 1)
    shutil.copy2(current_species, OUTPUT_DIR / current_species.name)
    best_species = current_species.with_name(f'{current_species.stem}_best{current_species.suffix}')
    if best_species.exists():
        shutil.copy2(best_species, OUTPUT_DIR / best_species.name)

    genus_result, species_result = evaluate_pair(current_genus, current_species)
    print_metrics(f'round {round_index}', genus_result, species_result)

    row = {
        'round': round_index,
        'genus_checkpoint': str(current_genus),
        'species_checkpoint': str(current_species),
        'top30_genus_accuracy': genus_result.accuracies[30],
        'top1_species_accuracy': species_result.accuracies[1],
        'top3_species_accuracy': species_result.accuracies[3],
        'top5_species_accuracy': species_result.accuracies[5],
        'plantclef_s': species_result.plantclef_s,
    }
    rows.append(row)

    final_genus = CHECKPOINT_DIR / 'final_scnn_genus_vgg16.pt'
    final_species = CHECKPOINT_DIR / 'final_scnn_species_vgg16.pt'
    shutil.copy2(current_genus, final_genus)
    shutil.copy2(current_species, final_species)
    shutil.copy2(final_genus, OUTPUT_DIR / final_genus.name)
    shutil.copy2(final_species, OUTPUT_DIR / final_species.name)

    with (OUTPUT_DIR / 'final_metrics.csv').open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    if row['top1_species_accuracy'] >= TARGET_HOLDOUT_TOP1 and row['top30_genus_accuracy'] >= TARGET_HOLDOUT_GENUS_TOP30:
        print('target reached on upper-bound holdout; stopping')
        break

print('saved final genus:', CHECKPOINT_DIR / 'final_scnn_genus_vgg16.pt')
print('saved final species:', CHECKPOINT_DIR / 'final_scnn_species_vgg16.pt')
print('saved Drive dir:', OUTPUT_DIR)

## 10. Final Diagnostic Evaluation And Artifacts

This writes predictions and plots for the adapted holdout run. Treat every artifact in this section as upper-bound diagnostics.

In [ ]:
%%bash
set -euo pipefail
cd /content/diploma

OUT=/content/drive/MyDrive/diploma_checkpoints/upper_bound/final_holdout_eval
mkdir -p "$OUT"

python -u -m plant_classifier.training.eval_species_cli \
  --config configs/leafscan_upper_bound.yaml \
  --query-config configs/leafscan_upper_bound.yaml \
  --genus-checkpoint checkpoints/final_scnn_genus_vgg16.pt \
  --species-checkpoint checkpoints/final_scnn_species_vgg16.pt \
  --reference-split adapt \
  --query-split holdout \
  --genus-references-per-genus 6 \
  --references-per-species 6 \
  --genus-candidates 30 \
  --genus-score-mode l1 \
  --species-score-mode l1 \
  --species-aggregation max \
  --genus-candidate-mode unique \
  --genus-weight-mode score \
  --top-k 1 3 5 \
  --output-dir "$OUT"

find "$OUT" -maxdepth 1 -type f -printf '%f\n' | sort